# Scikit Feature Tracking Algorithm -- Precipitation

### Import Packages

In [ ]:
# Import necessary libraries for identifying contours and plotting
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from skimage.measure import label, regionprops
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import cartopy.crs as ccrs

Load in the data using `xarray`, and extract the variable of interest.

We want to look at the total precipitation of the extra-tropical cyclones, which is stored under the `ETC_tp` variable.

In [ ]:
# Load in precipitation dataset
ds = xr.open_dataset('data/precip_ex.nc')

# Extra-tropical cyclone total precipitation
da = ds["ETC_tp"] 

We initialize the contour-detection parameters, and extract the latitude and longitude coordinates from the dataset. We then convert the coordinates into 2D grids so the found contours can be plotted in geographic space.

In [ ]:
# Define parameters for contour finding
threshold = 100000
min_area = 20
min_overlap = 0.2 

# Extract latitutde and longitude coordinates
lat = da[da.dims[1]].values # assumes dims like (time, lat, lon)
lon = da[da.dims[2]].values

# Make 2D grids for contour
lon2d, lat2d = np.meshgrid(lon, lat)

We define a function that detects connected regions with total precipitation levels below the threshold.

In [ ]:
def detect_features(
    field,
    threshold=threshold,
    min_area=min_area
):
    """
    Detect regions with a low total precipitation value (total precipitation <= threshold).
    """
    mask = field <= threshold
    labels = label(mask, connectivity=2)

    for lab in np.unique(labels)[1:]:
        if np.sum(labels == lab) < min_area:
            labels[labels == lab] = 0

    labels = label(labels > 0, connectivity=2)
    return labels

Our function identifies these low precipitation regions for a single time step. To track these regions over time, we must iterate through all time steps present in our data, and call `detect_features` on each interval.

In [ ]:
# Initialize features tracked through time
tracked = []
prev = {}
next_track_id = 1

# Loop through all timesteps
for t in range(da.sizes["time"]):
    field = da.isel(time=t).values # 2D data field for the current time step
    labels = detect_features(field, threshold=threshold, min_area=min_area) # feature-detection

    current = {}
    frame_info = []

    for r in regionprops(labels):
        mask = labels == r.label
        best_id = None
        best_overlap = 0.0

        for track_id, old_mask in prev.items():
            overlap = np.logical_and(mask, old_mask).sum() / mask.sum()
            if overlap > best_overlap:
                best_overlap = overlap
                best_id = track_id

        if best_overlap < min_overlap:
            best_id = next_track_id
            next_track_id += 1

        current[best_id] = mask
        frame_info.append({
            "track_id": best_id,
            "centroid_rc": r.centroid,
            "mask": mask
        })

    tracked.append(frame_info)
    prev = current

In the visualization, we will have a colorbar that represent precipitation levels. We want the bar to accurately depict the bulk of the data, so we limit the lower end to the 1st percentile of the data, and the upper end to the 99th percentile.

In [ ]:
# Find 1st percentile and 99th percentile of total precipitation values
vals = da.values
vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

levels = np.linspace(vmin, vmax, 16) # evenly spaced intervals
norm = mcolors.Normalize(vmin=vmin, vmax=vmax) # normalize
cmap = 'turbo' # colormap

We can animate the storm tracked over time by plotting the low precipitation regions at each timestep.

In [ ]:
def update(t):
    """
    Animation function, called once for each timestep in the dataset.
    """
    # Reset the map on each timestep
    ax.clear()
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.set_aspect('auto')

    field = da.isel(time=t).values # total precipitation data for current timestep

    # Colored contour lines
    cs = ax.contour(
        lon2d, lat2d, field,
        levels=levels,
        cmap=cmap,         
        linewidths=0.9,
        transform=ccrs.PlateCarree()
    )

    # tracked feature outlines + labels
    for feat in tracked[t]:
        mask = feat["mask"]

        ax.contour(
            lon2d, lat2d, mask.astype(int),
            levels=[0.5],
            colors="black",
            linewidths=1.6,
            transform=ccrs.PlateCarree()
        )

        y, x = feat["centroid_rc"]
        iy = int(round(y))
        ix = int(round(x))

        ax.plot(
            lon[ix], lat[iy],
            "ko", ms=3,
            transform=ccrs.PlateCarree()
        )

        ax.text(
            lon[ix], lat[iy],
            str(feat["track_id"]),
            fontsize=8,
            color="black",
            ha="left",
            va="bottom",
            transform=ccrs.PlateCarree(),
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1)
        )

    # Create outlines for continents/coastlines and gridlines
    ax.coastlines(color="0.55", linewidth=0.6)
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="0.5",
        alpha=0.25,
        linestyle="--"
    )
    
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(f"Tracked precipitation features | time index = {t}")

In [ ]:
# Create map
fig, ax = plt.subplots(figsize=(12, 4), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [lon.min(), lon.max(), lat.min(), lat.max()] # geographic boundaries of the map
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Create colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.15,
    orientation = 'horizontal'
)
cbar.set_label("ETC_p")

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=da.sizes["time"],
    interval=300,
    blit=False,
)

plt.close(fig)
HTML(anim.to_jshtml())